# Análisis Interseccional y de Brechas (Género e Interculturalidad)
Este notebook procesa las bases de datos raw para calcular los nuevos indicadores solicitados:
1. **Brechas de victimización y uso** (Género)
2. **Violencia digital específica CSEA** (Género)
3. **Control parental diferenciado** (Género)
4. **Autoidentificación cultural** (Interculturalidad)
5. **Lenguaje y contexto - Urbano vs Rural** (Interculturalidad)
6. **Acceso a la justicia y denuncia** (Interculturalidad)

Adicionalmente, estandariza **Dimensiones Demográficas** (Género, Área, Autoidentificación) para permitir cruces dinámicos entre Estudiantes, Padres y Docentes en Power BI.


In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Rutas de los archivos
PATH_ESTUDIANTES = '../raw_data/BBDD_Estudiantes_dash.xlsx'
PATH_PADRES = '../raw_data/BBDD_Padres_dash.xlsx'
PATH_DOCENTES = '../raw_data/BBDD_Docentes_dash.xlsx'

# Cargar las bases de datos
df_est = pd.read_excel(PATH_ESTUDIANTES)
df_pad = pd.read_excel(PATH_PADRES)
df_doc = pd.read_excel(PATH_DOCENTES)

print("Datos cargados correctamente:")
print(f"Estudiantes: {df_est.shape}")
print(f"Padres: {df_pad.shape}")
print(f"Docentes: {df_doc.shape}")


Datos cargados correctamente:
Estudiantes: (443, 69)
Padres: (330, 90)
Docentes: (187, 69)


## 1. Homologación de Dimensiones Maestras
Para cruzar datos entre Estudiantes, Padres y Docentes en Power BI sin complicaciones, vamos a crear columnas estandarizadas:
- `Poblacion`: Estudiante, Padre, Docente
- `Genero_Homologado`
- `Area_Homologada` (Urbano / Rural)
- `Identidad_Indigena` (Sí / No)

Vamos a explorar los valores únicos actuales para armar el mapeo.

In [5]:
# Extraer valores únicos para ver cómo homologar
print("--- ESTUDIANTES ---")
print("Género:", df_est['D2. ¿Género?'].unique())
print("Área:", df_est['D4. ¿Cómo describirías el lugar donde vives?'].unique())
print("Indígena:", df_est['Autoidentificación cultural'].unique())

print("\n--- PADRES ---")
print("Género:", df_pad['P2. ¿Cual es su Género?'].unique())
print("Área:", df_pad['P9. ¿En qué tipo de zona vive actualmente?'].unique())
print("Indígena:", df_pad['P7. ¿Se identifica como parte de alguna nación o pueblo indígena originario campesino?'].unique())

print("\n--- DOCENTES ---")
print("Género:", df_doc['D4. ¿Cuál es su género?'].unique())
print("Indígena:", df_doc['D5 ¿Se identifica como parte de alguna nación o pueblo indígena originario campesino?'].unique())


--- ESTUDIANTES ---
Género: <StringArray>
['Masculino', 'Femenino', 'No binario ', ' ', 'Prefiero no decirlo']
Length: 5, dtype: str
Área: <StringArray>
[                                      'Zona Urbana (Centro ciudad)',
 'Zona Periurbana o barrio periférico (Barrios alejados del centro)',
                                     'Zona Rural (campo, comunidad)']
Length: 3, dtype: str
Indígena: <StringArray>
['No', 'Si']
Length: 2, dtype: str

--- PADRES ---
Género: <StringArray>
['Femenino', 'Masculino']
Length: 2, dtype: str
Área: <StringArray>
[                                      'Zona urbana (Centro ciudad)',
 'Zona periurbana o barrio periférico (Barrios alejados del centro)',
                                     'Zona rural (Campo, comunidad)']
Length: 3, dtype: str
Indígena: <StringArray>
['No', 'Sí']
Length: 2, dtype: str

--- DOCENTES ---
Género: <StringArray>
['Femenino', 'Masculino']
Length: 2, dtype: str
Indígena: <StringArray>
['Sí', 'No']
Length: 2, dtype: str


## 2. Creación de Indicadores de Brechas (Binarios)
Ahora crearemos las variables lógicas para las dimensiones de Género e Interculturalidad basándonos en las preguntas específicas.

In [6]:
# --- ESTUDIANTES: Victimización y CSEA ---
col_fotos = '20. ¿Alguna vez sentiste presión para enviar fotos personales o íntimas a alguien por internet?'
col_burlas = '21. ¿Alguna vez recibiste comentarios sexualizados, burlas sobre tu cuerpo o tu apariencia a través de redes sociales o mensajes en internet?'

if col_fotos in df_est.columns and col_burlas in df_est.columns:
    print("\nRespuestas presión fotos:", df_est[col_fotos].dropna().unique()[:5])
    print("Respuestas burlas:", df_est[col_burlas].dropna().unique()[:5])
    
    # Rellenamos nulos con cadena vacía y usamos str.startswith vectorizado
    df_est['ind_presion_fotos'] = df_est[col_fotos].fillna('').astype(str).str.lower().str.startswith('si').astype(int)
    df_est['ind_burlas_sexualizadas'] = df_est[col_burlas].fillna('').astype(str).str.lower().str.startswith('si').astype(int)
    
    # Brecha de victimización (al menos uno de los dos)
    df_est['ind_victimizacion_digital'] = ((df_est['ind_presion_fotos'] == 1) | (df_est['ind_burlas_sexualizadas'] == 1)).astype(int)
    print("Victimización estudiantes calculada.")

# --- ESTUDIANTES: Acceso a Justicia ---
col_denuncia = '49. ¿Sabes a dónde puedes acudir para denunciar un caso de violencia digital en tu municipio?'
if col_denuncia in df_est.columns:
    df_est['ind_conoce_denuncia'] = df_est[col_denuncia].fillna('').astype(str).str.lower().str.startswith('si').astype(int)

# --- CONTROL PARENTAL DIFERENCIADO (Padres y Estudiantes) ---
col_control_pad = '17. ¿Las reglas o el tiempo de conexión a internet son iguales para sus hijos e hijas, independientemente de si son varones o mujeres?'
col_control_est1 = '18. ¿Tus padres/cuidadores controlan o supervisan el tiempo que pasas en internet de manera diferente según si eres varón o mujer?'

if col_control_pad in df_pad.columns:
    print("\nRespuestas reglas padres (diferenciado):", df_pad[col_control_pad].dropna().unique()[:5])
    df_pad['ind_control_diferenciado'] = df_pad[col_control_pad].fillna('').astype(str).str.lower().str.startswith('no').astype(int)

if col_control_est1 in df_est.columns:
    print("Respuestas reglas estudiantes (diferenciado):", df_est[col_control_est1].dropna().unique()[:5])
    df_est['ind_control_diferenciado'] = df_est[col_control_est1].fillna('').astype(str).str.lower().str.startswith('si').astype(int)



Respuestas presión fotos: <StringArray>
[                                                           'No, nunca',
                                           'Si, una vez, pero me negué',
 'La verdad no se entiende la pregunta, pero nunca envié nada a nadie ',
                                            'Nunca mande fotos a nadie',
                              'Nunca mande fotos personales o intimas,']
Length: 5, dtype: str
Respuestas burlas: <StringArray>
['No, nunca', 'Si, una o dos veces', 'Prefiero no responder',
 'Si, varias veces']
Length: 4, dtype: str
Victimización estudiantes calculada.

Respuestas reglas padres (diferenciado): <StringArray>
[          'Sí, las reglas son iguales para todos/as',
                                          'No aplica',
    'No, los varones tienen más libertad de conexión',
 'No, las niñas tienen más restricciones de conexión']
Length: 4, dtype: str
Respuestas reglas estudiantes (diferenciado): <StringArray>
[                'Sí, a las mujeres las

## 3. Apilación y Exportación (Tabla de Hechos)
Ahora vamos a estandarizar los nombres de las columnas clave y usar `pd.concat` para apilar verticalmente las tres bases de datos. 
Esto generará una tabla unificada con una columna **`Poblacion`**, permitiéndote hacer comparaciones directas en Power BI con un simple filtro (slicer).

In [7]:
import os

# --- 1. Seleccionar y renombrar columnas Estudiantes ---
df_est_clean = pd.DataFrame({
    'Poblacion': 'Estudiantes',
    'Municipio': df_est.get('Municipio', 'Desconocido'),
    'Genero': df_est.get('D2. ¿Género?', 'Desconocido'),
    'Area': df_est.get('D4. ¿Cómo describirías el lugar donde vives?', 'Desconocido'),
    'Identidad_Indigena': df_est.get('Autoidentificación cultural', 'Desconocido'),
    'Ind_Victimizacion_Digital': df_est.get('ind_victimizacion_digital', np.nan),
    'Ind_Control_Diferenciado': df_est.get('ind_control_diferenciado', np.nan),
    'Ind_Conoce_Denuncia': df_est.get('ind_conoce_denuncia', np.nan)
})

# --- 2. Seleccionar y renombrar columnas Padres ---
df_pad_clean = pd.DataFrame({
    'Poblacion': 'Padres',
    'Municipio': df_pad.get('Municipio', 'Desconocido'),
    'Genero': df_pad.get('P2. ¿Cual es su Género?', 'Desconocido'),
    'Area': df_pad.get('P9. ¿En qué tipo de zona vive actualmente?', 'Desconocido'),
    'Identidad_Indigena': df_pad.get('P7. ¿Se identifica como parte de alguna nación o pueblo indígena originario campesino?', 'Desconocido'),
    'Ind_Victimizacion_Digital': np.nan, # Esta variable específica de prevalencia fue medida en estudiantes
    'Ind_Control_Diferenciado': df_pad.get('ind_control_diferenciado', np.nan),
    'Ind_Conoce_Denuncia': np.nan 
})

# --- 3. Seleccionar y renombrar columnas Docentes ---
df_doc_clean = pd.DataFrame({
    'Poblacion': 'Docentes',
    'Municipio': df_doc.get('Municipio', 'Desconocido'),
    'Genero': df_doc.get('D4. ¿Cuál es su género?', 'Desconocido'),
    'Area': 'No medido en docentes', # Docentes no tienen columna de área específica en el form
    'Identidad_Indigena': df_doc.get('D5 ¿Se identifica como parte de alguna nación o pueblo indígena originario campesino?', 'Desconocido'),
    'Ind_Victimizacion_Digital': np.nan,
    'Ind_Control_Diferenciado': np.nan,
    'Ind_Conoce_Denuncia': np.nan
})

# --- 4. Unificar Bases ---
df_consolidado = pd.concat([df_est_clean, df_pad_clean, df_doc_clean], ignore_index=True)

# Homologación rápida de Área para que los filtros de Power BI queden limpios
df_consolidado['Area'] = df_consolidado['Area'].fillna('Desconocido').astype(str)
df_consolidado['Area'] = np.where(df_consolidado['Area'].str.contains('Urbana|Ciudad', case=False, na=False), 'Urbano', 
                         np.where(df_consolidado['Area'].str.contains('Rural|Campo', case=False, na=False), 'Rural', 
                         np.where(df_consolidado['Area'].str.contains('Periurban', case=False, na=False), 'Periurbano', 
                                  df_consolidado['Area'])))

# Homologación rápida de Identidad Indígena
df_consolidado['Identidad_Indigena'] = df_consolidado['Identidad_Indigena'].fillna('No').astype(str).str.strip().str.title()
df_consolidado['Identidad_Indigena'] = np.where(df_consolidado['Identidad_Indigena'].str.startswith('Si'), 'Sí', 'No')

# --- 5. Exportar ---
os.makedirs('../output', exist_ok=True)
ruta_exportacion = '../output/BBDD_Brechas_Consolidado.csv'
df_consolidado.to_csv(ruta_exportacion, index=False, encoding='utf-8-sig')

print(f"¡Tabla de Hechos unificada con éxito! Exportada a: {ruta_exportacion}")
print(f"Total de registros apilados: {df_consolidado.shape[0]} (Estudiantes, Padres y Docentes)")
df_consolidado.sample(5)


¡Tabla de Hechos unificada con éxito! Exportada a: ../output/BBDD_Brechas_Consolidado.csv
Total de registros apilados: 960 (Estudiantes, Padres y Docentes)


,Poblacion,Municipio,Genero,Area,Identidad_Indigena,Ind_Victimizacion_Digital,Ind_Control_Diferenciado,Ind_Conoce_Denuncia
882,Docentes,Villa Montes,Masculino,No medido en docentes,No,NaN,NaN,NaN
868,Docentes,Palca,Masculino,No medido en docentes,No,NaN,NaN,NaN
366,Estudiantes,Villa Montes,Femenino,Urbano,No,0.0,0.0,0.0
874,Docentes,Palca,Femenino,No medido en docentes,No,NaN,NaN,NaN
467,Padres,La Paz,Masculino,Urbano,No,NaN,0.0,NaN
